# Kochi-LLM v1.0 — publish checkpoints to Hugging Face

Converts the **most-trained** checkpoint of every attached Kaggle run to bf16 safetensors and pushes each
one into its own folder on `main`:

```
runs/
├── index.json          manifest of every published run
├── run-01/             earliest run by step count
│   ├── model.safetensors
│   ├── config.json
│   └── metadata.json
├── run-02/
└── ...
```

Folders are tagged **chronologically by step count** (`run-01` = earliest, `run-NN` = latest), *not* by
Kaggle slug — `trainning-3` and `trainning-3-different-dataset` both end in `3` and would collide.

Each run is committed separately and lands in its own folder, so this notebook is **order-independent and
safe to re-run**: a repeat overwrites only that run's folder. `runs/index.json` merges across sessions,
so you can publish a few runs at a time and the manifest keeps accumulating.

**Scope: weights only.** The model card, `ARCHITECTURE.md`, `TRAINING.md`, `SAMPLES.md`, the loss-curve
assets, `LICENSE`, and the source files are already published at the repo root. This notebook will not
overwrite an existing `README.md`.

## Why you may need several sessions

Each checkpoint is 4.1 GB, so attaching all nine at once (~37 GB) will exceed Kaggle's notebook input
limit. Attach as many as the limit allows, run this notebook, then attach the next batch and run it
again.

## Setup

- Attach one or more Kaggle notebook outputs containing `best_model.pth` / `latest_backup_model.pth`
- **Internet ON**. No GPU needed — all work is CPU + network
- Add an `HF_TOKEN` Kaggle secret with **write** access


In [ ]:
%%bash
set -e
# No -U: forcing the newest huggingface_hub breaks the image's gradio /
# transformers / datasets pins for no benefit. We only need safetensors.
pip install -q safetensors
python -c "import huggingface_hub, safetensors; print('huggingface_hub', huggingface_hub.__version__); print('safetensors', safetensors.__version__)"

DIRS=$(find /kaggle/input \( -name 'best_model.pth' -o -name 'latest_backup_model.pth' \) | xargs -r -n1 dirname | sort -u)
echo "$DIRS"
N=$(printf '%s' "$DIRS" | grep -c . || true)
echo "found $N run dir(s)"
[ "$N" -eq 0 ] && { echo 'NO RUNS ATTACHED'; ls -R /kaggle/input | head -40; exit 1; }

PY_SRC=''
for d in $DIRS /kaggle/input/datasets/dadhichisarker/kochi-llm/model; do
  if [ -f "$d/architecture.py" ]; then PY_SRC=$d; break; fi
done
[ -z "$PY_SRC" ] && { echo 'NO PY SOURCE'; exit 1; }
cp "$PY_SRC"/*.py /kaggle/working/
find /kaggle/input -name 'custom_bpe_tokenizer.pkl' -exec cp {} /kaggle/working/ \; 2>/dev/null || true
ls -lh /kaggle/working/*.py /kaggle/working/*.pkl 2>/dev/null || true


## Preflight

Guards against `huggingface_hub` API drift. In 0.36 several methods made their trailing parameters
keyword-only, which turns a positional call into a runtime `TypeError` — previously this would have
failed only *after* multi-GB uploads were staged. Verify the interpreter before doing any work.


In [ ]:
import inspect
from huggingface_hub import HfApi, CommitOperationAdd

REQUIRED = {
    'upload_file':      ['path_or_fileobj', 'path_in_repo', 'repo_id', 'commit_message', 'revision'],
    'create_commit':    ['repo_id', 'operations', 'commit_message', 'revision'],
    'create_repo':      ['repo_id', 'repo_type', 'exist_ok'],
    'whoami':           [],
    'repo_info':        ['repo_id'],
    'list_repo_files':  ['repo_id'],
}

import huggingface_hub
print('huggingface_hub', huggingface_hub.__version__)

bad = []
for meth, params in REQUIRED.items():
    sig = inspect.signature(getattr(HfApi, meth))
    names = set(sig.parameters)
    missing = [p for p in params if p not in names]
    if missing:
        bad.append('%s() missing %s' % (meth, missing))
    for p in params:
        if p in names and sig.parameters[p].kind is inspect.Parameter.POSITIONAL_ONLY:
            bad.append('%s(): %s is positional-only' % (meth, p))

ops_sig = set(inspect.signature(CommitOperationAdd.__init__).parameters)
for p in ('path_in_repo', 'path_or_fileobj'):
    if p not in ops_sig:
        bad.append('CommitOperationAdd missing %s' % p)

if bad:
    raise SystemExit('huggingface_hub API drift:\n  ' + '\n  '.join(bad))
print('preflight OK - all hub calls in this notebook bind')


## Upload plan

Check this table before anything ships.

`chosen` is the checkpoint with the **highest step count**, not necessarily `best_model.pth`: several
early runs carry a stale Phase-1 `best_model.pth` while their real weights sit in
`latest_backup_model.pth`.


In [ ]:
import glob, os, re
import torch

WEIGHTS = ('best_model.pth', 'latest_backup_model.pth')

dirs = sorted({os.path.dirname(p)
               for p in glob.glob('/kaggle/input/**/*.pth', recursive=True)
               if os.path.basename(p) in WEIGHTS})

plan = []
for d in dirs:
    cands = []
    for n in WEIGHTS:
        fp = os.path.join(d, n)
        if os.path.exists(fp):
            ck = torch.load(fp, map_location='cpu', weights_only=False, mmap=True)
            cands.append((int(ck.get('iter', -1)), fp, ck))
    if cands:
        it, fp, ck = max(cands, key=lambda t: t[0])
        plan.append({'dir': d, 'slug': os.path.basename(d.rstrip('/')),
                     'iter': it, 'path': fp, 'ckpt': ck})

# tag chronologically by step count: run-01 is the earliest run, run-NN the latest.
# Deriving the tag from the Kaggle slug collides (trainning-3 and
# trainning-3-different-dataset both end in 3) and does not sort meaningfully.
plan.sort(key=lambda e: e['iter'])
for i, e in enumerate(plan, 1):
    e['branch'] = 'run-%02d' % i

print('%-36s %-8s %7s %8s %9s  %s' % ('source run', 'branch', 'iter', 'val', 'tokens', 'chosen file'))
for e in plan:
    v = e['ckpt'].get('val_loss')
    v = float(v) if v is not None else float('nan')
    print('%-36s %-8s %7d %8.4f %9.1fM  %s' % (
        e['slug'][:36], e['branch'], e['iter'], v, e['iter'] * 32768 / 1e6,
        os.path.basename(e['path'])))
print('\n%d run(s) will be pushed.' % len(plan))
print('note: val_loss < 5 on early runs is the stale Phase-1 Alpaca baseline,')
print('      not a Phase-2 score. metadata.json flags this via val_loss_is_phase2.')


In [ ]:
import os
from huggingface_hub import HfApi

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no Kaggle secret:', e)
if not TOKEN:
    TOKEN = os.environ.get('HF_TOKEN')
if not TOKEN:
    raise SystemExit('Add an HF_TOKEN Kaggle secret (write access) and re-run.')

api = HfApi(token=TOKEN)
USER = api.whoami()['name']
REPO = USER + '/KochiLLM-v1.0'
api.create_repo(REPO, repo_type='model', exist_ok=True)
print('target repo:', REPO)

CODE_FILES = ['architecture.py', 'tokenizer.py', 'data_loader.py',
              'generate.py', 'prepare_data.py', 'train_custom.py',
              'custom_bpe_tokenizer.pkl']

existing = {s.rfilename for s in (api.repo_info(REPO).siblings or [])}
print('repo already has %d file(s)' % len(existing))

added = [f for f in CODE_FILES if os.path.exists(f) and f not in existing]
for f in added:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO,
                    commit_message='Add ' + f)
print('uploaded code:', added or 'nothing (already present)')

if 'README.md' in existing:
    print('README.md already published - leaving it untouched.')
else:
    import json as _json
    with open('README.md', 'w', encoding='utf-8') as f:
        f.write(_json.loads("---\nlibrary_name: pytorch\ntags:\n  - moe\n  - mixture-of-experts\n  - gqa\n  - multi-token-prediction\n  - from-scratch\n  - custom-architecture\nlanguage:\n  - en\nlicense: mit\n---\n\n# KochiLLM v1.0\n\nA **1.02B-parameter Mixture-of-Experts language model trained entirely from scratch** on 2x Tesla T4\n(~52 GPU-hours), including its own BPE tokenizer, data pipeline, and distributed training setup.\n\nFinal validation loss **3.96** after exactly one epoch over 180.2M tokens of FineWeb-Edu +\nCosmopedia-v2 (0.176 tokens/param). Weights are bf16.\n\n## Loading\n\nNot an `AutoModel` release - the architecture is custom code and the tokenizer is a custom pickle.\nWeights live in per-run folders; see `runs/index.json` for the manifest.\n\n```python\nimport json, torch\nfrom huggingface_hub import hf_hub_download\nfrom safetensors.torch import load_file\nfrom architecture import MyCustomLLM\nfrom tokenizer import MyCustomTokenizer\n\nREPO, TAG = 'ShayonSarker/KochiLLM-v1.0', 'run-09'\n\nconfig = json.load(open(hf_hub_download(REPO, 'runs/%s/config.json' % TAG, token=...)))\ntok = MyCustomTokenizer(vocab_size=config['vocab_size'])\ntok.load('custom_bpe_tokenizer.pkl')\n\nmodel = MyCustomLLM(**config)                 # re-ties embed.weight <-> output.weight\nsd = load_file(hf_hub_download(REPO, 'runs/%s/model.safetensors' % TAG, token=...))\nsd['output.weight'] = sd['embed.weight']      # no-op here: FSDP emits both, byte-identical\nmodel.load_state_dict(sd, strict=False)\nmodel.eval().cuda().to(torch.bfloat16)\n```\n\n## Limitations\n\n**No factual knowledge.** At 0.176 tokens/param the model learned fluency and style, not facts;\ngenerated names and dates are plausible and wrong. **Not instruction-tuned** - it continues documents\nrather than answering questions. Context is 1024 tokens. Validation was in-distribution. Full\narchitecture and training documentation is in `ARCHITECTURE.md` and `TRAINING.md`.\n\n## Source\n\n<https://github.com/Dadhichi-Sarker-Shayon/KochiLLM-v1.0>\n\nMIT licensed.\n"))
    api.upload_file(path_or_fileobj='README.md', path_in_repo='README.md',
                    repo_id=REPO, commit_message='Add fallback model card')
    print('wrote fallback model card (no README existed)')


## Notes

- **`runs/index.json`** is a manifest of every published run: step count, tokens seen, validation loss,
  whether that loss is a genuine Phase-2 reading, the source Kaggle run, and the folder path. It merges
  with whatever earlier sessions wrote, so batches accumulate.
- **`val_loss_is_phase2` matters.** Checkpoints from before the Phase-2 best-tracking fix carry the
  Phase-1 Alpaca loss (3.8501) in `val_loss` and have no `latest_val_loss`. Read that flag to tell them
  apart; for those runs the honest statement is "step N of a run whose Phase-2 validation loss was not
  separately recorded".
- **Tied weights.** `embed.weight` and `output.weight` are the same tensor in the model, but the FSDP
  full-state-dict gather materialises them as two separate arrays, so **both are present** in each
  `model.safetensors` and are byte-identical. `load_state_dict` works with `strict=True`. The
  `sd['output.weight'] = sd['embed.weight']` line is a no-op for FSDP checkpoints and additionally
  supports any checkpoint saved without FSDP. `tied_weights_dropped` is therefore usually empty.
- **bf16.** Cast down from the fp32 checkpoint, halving size to ~2.05 GB. Intended for inference, not
  for resuming training.
- **`config.json` stays pure.** Written verbatim from the checkpoint so `MyCustomLLM(**config)` keeps
  working. Do not add `model_type` / `architectures` keys to it.
- **Not an `AutoModel` release.** Custom architecture and a custom pickle tokenizer; see the model card.
- **A failing run does not abort the batch.** Each run is attempted independently; failures are
  reported together at the end so one bad checkpoint cannot waste the other uploads.


In [ ]:
import gc, json, math, os
import torch
from safetensors.torch import save_file
from huggingface_hub import CommitOperationAdd, hf_hub_download

TMP = ('model.safetensors', 'config.json', 'metadata.json')
RUNS_DIR = 'runs'


def to_bf16(v):
    # explicit new allocation: .to(bfloat16) is a no-op when v is already bf16,
    # which would leave the result aliasing the mmap'd checkpoint storage
    out = torch.empty(v.shape, dtype=torch.bfloat16)
    out.copy_(v)
    return out


def build_files(ckpt, entry):
    sd = ckpt['model_state_dict']
    seen, kept, dropped = set(), {}, []
    for k, v in sd.items():
        if v.data_ptr() in seen:
            dropped.append(k)
            continue
        seen.add(v.data_ptr())
        kept[k] = to_bf16(v)

    save_file(kept, 'model.safetensors', metadata={'format': 'pt'})

    with open('config.json', 'w', encoding='utf-8') as f:
        json.dump(ckpt.get('config', {}), f, indent=2)

    latest = float(ckpt.get('latest_val_loss', float('nan')))
    meta = {
        'iter': int(ckpt.get('iter', -1)),
        'val_loss': float(ckpt.get('val_loss', float('nan'))),
        'latest_val_loss': latest,
        # checkpoints predating the Phase-2 reset carry the stale Alpaca baseline
        'val_loss_is_phase2': not math.isnan(latest),
        'source_run': entry['slug'],
        'dtype': 'bfloat16',
        'tied_weights_dropped': dropped,
        'tied_weights_shared_storage': bool(dropped),
        # FSDP's full-state-dict gather materialises the aliased embed/output weights as
        # two separate arrays, so `dropped` is normally empty and both are byte-identical.
        'note': ('embed.weight and output.weight are the same tensor in the model, but the FSDP '
                 'full-state-dict gather materialises them as two separate arrays. Both are '
                 'present in model.safetensors and are byte-identical.'),
    }
    with open('metadata.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)
    return meta


def make_ops(subdir):
    # MUST be a fresh list per commit: huggingface_hub >= 2.0 marks operations as
    # consumed after a commit and raises on reuse ("has already being committed").
    return [CommitOperationAdd(path_in_repo='%s/%s' % (subdir, n), path_or_fileobj=n)
            for n in TMP]


# Merge into any manifest from earlier sessions so repeated Kaggle runs accumulate
# instead of clobbering.
index = {}
try:
    index = json.load(open(hf_hub_download(REPO, '%s/index.json' % RUNS_DIR,
                                           revision='main', token=TOKEN)))
    if not isinstance(index, dict):
        index = {}
except Exception:
    index = {}

print('manifest already lists %d run(s)' % len(index))
print('pushing %d run(s) to %s under %s/' % (len(plan), REPO, RUNS_DIR))

failures = []
for e in plan:
    sub = '%s/%s' % (RUNS_DIR, e['branch'])
    try:
        meta = build_files(e['ckpt'], e)
        val = meta['val_loss']
        msg = '%s: iter %d, val %.4f (from %s)' % (sub, e['iter'], val, e['slug'])

        api.create_commit(REPO, operations=make_ops(sub), commit_message=msg,
                          revision='main')
        print('pushed', sub, '| iter', e['iter'], '| val %.4f' % val)

        index[e['branch']] = {
            'iter': e['iter'],
            'tokens_seen': e['iter'] * 32768,
            'val_loss': val,
            'val_loss_is_phase2': meta['val_loss_is_phase2'],
            'source_run': e['slug'],
            'path': sub,
        }
    except Exception as ex:
        failures.append((sub, repr(ex)))
        print('  FAILED %s -> %s' % (sub, ex))
    finally:
        # Drop the mmap reference before moving on. With all nine runs attached the
        # plan holds nine lazily-mapped checkpoints; each one is paged fully into
        # RAM as we convert it, so keeping every mapping alive would push resident
        # memory toward 9 x 4.1 GB and risk an OOM kill mid-batch.
        e['ckpt'] = None
        for n in TMP:
            if os.path.exists(n):
                os.remove(n)
        gc.collect()

if not failures:
    with open('index.json', 'w', encoding='utf-8') as f:
        json.dump({'model': 'KochiLLM-v1.0',
                   'final_run': plan[-1]['branch'] if plan else None,
                   'runs': dict(sorted(index.items()))}, f, indent=2)
    api.upload_file(path_or_fileobj='index.json', path_in_repo='%s/index.json' % RUNS_DIR,
                    repo_id=REPO, revision='main',
                    commit_message='Update runs/index.json (%d run(s))' % len(index))
    print('\nupdated %s/index.json with %d run(s)' % (RUNS_DIR, len(index)))

if failures:
    print('\n%d run(s) FAILED:' % len(failures))
    for b, err in failures:
        print('  ', b, err)
    raise SystemExit(1)
print('done')


In [ ]:
files = api.list_repo_files(REPO)
runs = sorted(f for f in files if f.startswith('runs/'))
print('repo:', REPO)
print('root files: %d' % len([f for f in files if '/' not in f]))
print('\ncheckpoint tree:')
for f in runs:
    print('  ', f)
